In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

In [ ]:
# データの読み込み（国立がん研究センター がん統計 全国がん登録）
df_raw = pd.read_excel('data/raw/cancer_incidenceNCR(2016-2023).xls', sheet_name='rate')
df_raw.head(3)

In [ ]:
df_raw.shape

In [ ]:
# 部位の確認
df_raw['部位'].unique()

In [ ]:
# 性別の確認
df_raw['性別'].unique()

In [ ]:
# 使う部位（上皮内がんを含む集計は使わない、結腸と直腸は大腸を使う）
sites = {1: '全部位', 3: '口腔・咽頭', 4: '食道', 5: '胃', 67: '大腸', 8: '肝臓', 9: '胆嚢・胆管',
         10: '膵臓', 12: '肺', 14: '乳房', 17: '子宮頸部', 18: '子宮体部', 19: '卵巣', 20: '前立腺',
         21: '膀胱', 22: '腎・尿路（膀胱除く）', 24: '甲状腺', 25: '悪性リンパ腫', 27: '白血病'}

In [ ]:
# 男女別のデータだけ使う（総数は使わない）
df = df_raw[df_raw['コード'].isin(sites.keys())]
df = df[df['性別'] != '総数']
df.shape

In [ ]:
# 年齢の列（85歳以上はまとめた列を使う）
age_cols = ['0-4歳', '5-9歳', '10-14歳', '15-19歳', '20-24歳', '25-29歳', '30-34歳', '35-39歳',
            '40-44歳', '45-49歳', '50-54歳', '55-59歳', '60-64歳', '65-69歳', '70-74歳',
            '75-79歳', '80-84歳', '85歳以上（再掲）']

In [ ]:
# 年齢を列から行にする（1行 = 部位・性別・年・年齢）
df = df.melt(id_vars=['コード', '性別', '診断年'], value_vars=age_cols, var_name='age_group', value_name='rate')
df.head(3)

In [ ]:
df.shape

In [ ]:
# 欠損値の確認
df.isnull().sum()

In [ ]:
df['rate'] = df['rate'].astype(float)

In [ ]:
# 年齢を数値にする
age_map = {'0-4歳': 0, '5-9歳': 5, '10-14歳': 10, '15-19歳': 15, '20-24歳': 20, '25-29歳': 25,
           '30-34歳': 30, '35-39歳': 35, '40-44歳': 40, '45-49歳': 45, '50-54歳': 50, '55-59歳': 55,
           '60-64歳': 60, '65-69歳': 65, '70-74歳': 70, '75-79歳': 75, '80-84歳': 80, '85歳以上（再掲）': 85}
df['age'] = df['age_group'].map(age_map)

# 性別を数値にする（男:1 女:0）
df['sex'] = df['性別'].map({'男': 1, '女': 0})

df = df.rename(columns={'コード': 'site', '診断年': 'year'})
df = df[['year', 'sex', 'site', 'age', 'rate']]
df.head(3)

In [ ]:
df.to_csv('data/incidence.csv', index=False)

In [ ]:
# データ統計量の確認
df.describe()

In [ ]:
# 年齢ごとの罹患率（2023年 男性）
d = df[(df['year'] == 2023) & (df['sex'] == 1)]
for site in [5, 67, 12, 20]:
    plt.plot(d[d['site'] == site]['age'], d[d['site'] == site]['rate'])
plt.xlabel('age')
plt.ylabel('rate');

In [ ]:
# 年ごとの変化（全部位 60-64歳 男性）
d = df[(df['site'] == 1) & (df['age'] == 60) & (df['sex'] == 1)]
plt.plot(d['year'], d['rate'])
plt.xlabel('year')
plt.ylabel('rate');

In [ ]:
# 罹患率の分布
plt.hist(df['rate'], bins=50);

In [ ]:
# 高齢ほど値がかなり大きいので log をとる
plt.hist(np.log(df['rate'] + 1), bins=50);

In [ ]:
# 入力変数と出力変数の切り分け（部位はダミー変数にする）
x = pd.get_dummies(df[['year', 'sex', 'age', 'site']], columns=['site'], dtype=int)
t = np.log(df['rate'] + 1)

x.shape, t.shape

In [ ]:
# 2016〜2022年で学習して、2023年を予測できるか確認する
x_train = x[df['year'] <= 2022]
t_train = t[df['year'] <= 2022]
x_test = x[df['year'] == 2023]
t_test = t[df['year'] == 2023]

x_train.shape, x_test.shape

In [ ]:
from sklearn import metrics

# 罹患率に戻した値で比べる
rate_test = np.exp(t_test) - 1

In [ ]:
# 比較用：2022年の値をそのまま 2023年の予測にする
y_base = df[df['year'] == 2022]['rate'].values
print('MAE : ', metrics.mean_absolute_error(rate_test, y_base))

In [ ]:
# モデルの定義
from sklearn.linear_model import LinearRegression
model = LinearRegression()

In [ ]:
# モデルの学習
model.fit(x_train, t_train)

In [ ]:
# モデルの評価
print('train score : ', model.score(x_train, t_train))
print('test score : ', model.score(x_test, t_test))

y = np.exp(model.predict(x_test)) - 1
print('MAE : ', metrics.mean_absolute_error(rate_test, y))

In [ ]:
# モデルの定義
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, random_state=0)

In [ ]:
# モデルの学習
model.fit(x_train, t_train)

In [ ]:
# モデルの評価
print('train score : ', model.score(x_train, t_train))
print('test score : ', model.score(x_test, t_test))

y = np.exp(model.predict(x_test)) - 1
print('MAE : ', metrics.mean_absolute_error(rate_test, y))

In [ ]:
# モデルの定義
from sklearn.ensemble import GradientBoostingRegressor
model = GradientBoostingRegressor(n_estimators=1000, max_depth=6, learning_rate=0.05, random_state=0)

In [ ]:
# モデルの学習
model.fit(x_train, t_train)

In [ ]:
# モデルの評価
print('train score : ', model.score(x_train, t_train))
print('test score : ', model.score(x_test, t_test))

y = np.exp(model.predict(x_test)) - 1
print('MAE : ', metrics.mean_absolute_error(rate_test, y))

In [ ]:
# max_depth を変えてみる
for depth in [4, 6, 8]:
    model_test = GradientBoostingRegressor(n_estimators=500, max_depth=depth, learning_rate=0.05, random_state=0)
    model_test.fit(x_train, t_train)
    y_test = np.exp(model_test.predict(x_test)) - 1
    print(depth, metrics.mean_absolute_error(rate_test, y_test))

In [ ]:
# 予測値と実測値
plt.scatter(t_test, model.predict(x_test), s=5)
plt.xlabel('t_test')
plt.ylabel('predict');

In [ ]:
# 特徴量重要度の確認
plt.figure(figsize=(7, 8))
plt.barh(y=x.columns, width=model.feature_importances_);

In [ ]:
# 全部の年で学習し直す
model = GradientBoostingRegressor(n_estimators=1000, max_depth=6, learning_rate=0.05, random_state=0)
model.fit(x, t)

In [ ]:
import pickle

# モデルの保存
with open('models/model.pkl', 'wb') as f:
    pickle.dump(model, f)